In [1]:
# =========================
# CELL 1 — INSTALL
# =========================

!pip install groq google-generativeai gradio pandas openpyxl

In [2]:
# =========================
# CELL 2 — IMPORTS
# =========================

import json
import sys

import pandas as pd
import gradio as gr

from groq import Groq

import google.generativeai as genai

from google.colab import userdata

from IPython.display import display

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# =========================
# CELL 3 — API KEYS
# =========================

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [4]:
# =========================
# CELL 4 — CLIENTS
# =========================

groq_client = Groq(
    api_key=GROQ_API_KEY
)

genai.configure(
    api_key=GEMINI_API_KEY
)

In [5]:
# =========================
# CELL 5 — MODELS
# =========================

MODELS = {

    # =====================
    # GROQ MODELS
    # =====================

    "llama": {
        "provider": "groq",
        "model": "llama-3.3-70b-versatile"
    },

    # =====================
    # GEMINI MODELS
    # =====================

    "gemini-flash": {
        "provider": "gemini",
        "model": "gemini-2.5-flash"
    },

    "gemini-pro": {
        "provider": "gemini",
        "model": "gemini-2.5-pro"
    }
}

In [6]:
# =========================
# CELL 6 — DATASET GENERATOR
# =========================

def generate_dataset(topic, rows, model_name):

    prompt = f"""
    You are a dataset generator.

    Generate {rows} rows of synthetic data about:
    {topic}

    Rules:
    - Return ONLY valid JSON
    - No markdown
    - No explanation
    - Output must start with [
    """

    provider = MODELS[model_name]["provider"]

    selected_model = MODELS[model_name]["model"]

    try:

        # =====================
        # GROQ
        # =====================

        if provider == "groq":

            response = groq_client.chat.completions.create(
                model=selected_model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=1
            )

            content = response.choices[0].message.content

        # =====================
        # GEMINI
        # =====================

        elif provider == "gemini":

            model = genai.GenerativeModel(
                selected_model
            )

            response = model.generate_content(
                prompt
            )

            content = response.text

        # =====================
        # CLEAN JSON
        # =====================

        content = content.replace(
            "```json",
            ""
        )

        content = content.replace(
            "```",
            ""
        )

        content = content.strip()

        # =====================
        # LOAD JSON
        # =====================

        data = json.loads(content)

        df = pd.DataFrame(data)

        # =====================
        # NOTEBOOK OUTPUT
        # =====================

        print("\n========== GENERATED DATASET ==========\n")

        print(f"MODEL USED: {model_name}")

        print(f"\nTOTAL ROWS: {len(df)}")

        print("\nCOLUMNS:\n")

        print(df.columns.tolist())

        print("\nFIRST 5 ROWS:\n")

        display(df.head())

        print("\n=======================================\n")

        sys.stdout.flush()

        # =====================
        # SAVE FILES
        # =====================

        csv_file = "dataset.csv"

        json_file = "dataset.json"

        excel_file = "dataset.xlsx"

        df.to_csv(
            csv_file,
            index=False
        )

        df.to_json(
            json_file,
            orient="records",
            indent=2
        )

        df.to_excel(
            excel_file,
            index=False
        )

        return (
            df,
            csv_file,
            json_file,
            excel_file
        )

    except Exception as e:

        print("\n========== ERROR ==========\n")

        print(str(e))

        if "content" in locals():

            print("\nRAW OUTPUT:\n")

            print(content)

        print("\n===========================\n")

        sys.stdout.flush()

        error_df = pd.DataFrame({
            "Error": [str(e)]
        })

        return (
            error_df,
            None,
            None,
            None
        )

In [7]:
# =========================
# CELL 7 — GRADIO UI
# =========================

demo = gr.Interface(

    fn=generate_dataset,

    inputs=[

        gr.Textbox(
            label="Dataset Topic",
            placeholder="Example: Hospital Patients"
        ),

        gr.Slider(
            1,
            50,
            value=10,
            step=1,
            label="Rows"
        ),

        gr.Dropdown(
            choices=[
                "llama",
                "gemini-flash",
                "gemini-pro"
            ],

            value="llama",

            label="Model"
        )
    ],

    outputs=[

        gr.Dataframe(
            label="Generated Dataset"
        ),

        gr.File(
            label="Download CSV"
        ),

        gr.File(
            label="Download JSON"
        ),

        gr.File(
            label="Download Excel"
        )
    ],

    title="AI Synthetic Dataset Generator",

    description="""
    Generate synthetic datasets using:

    • Groq Llama
    • Gemini Flash
    • Gemini Pro
    """,

    allow_flagging="never"
)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


In [8]:
gr.close_all()

demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9f753d8844dd44ca9d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



========== GENERATED DATASET ==========

MODEL USED: llama

TOTAL ROWS: 10

COLUMNS:

['id', 'title', 'content', 'label']

FIRST 5 ROWS:



,id,title,content,label
0,1,Breaking News,A new study shows...,Real
1,2,You Won't Believe,Click here to learn more...,Fake
2,3,Scientific Discovery,Researchers have found...,Real
3,4,Exclusive Report,A source close to the matter...,Fake
4,5,Economic Update,The latest numbers show...,Real





========== GENERATED DATASET ==========

MODEL USED: gemini-flash

TOTAL ROWS: 30

COLUMNS:

['user_id', 'registration_date', 'subscription_type', 'monthly_spending_usd', 'last_active_date', 'total_watch_hours_month', 'average_session_duration_minutes', 'num_unique_titles_watched_month', 'most_watched_genre', 'device_preference', 'country', 'age_group', 'had_billing_issues_last_6_months', 'num_profiles_on_account', 'churn_risk_score']

FIRST 5 ROWS:



,user_id,registration_date,subscription_type,monthly_spending_usd,last_active_date,total_watch_hours_month,average_session_duration_minutes,num_unique_titles_watched_month,most_watched_genre,device_preference,country,age_group,had_billing_issues_last_6_months,num_profiles_on_account,churn_risk_score
0,usr_2d4a1b9e-0e2c-4f7d-8a1b-9e0e2c4f7d8a,2020-05-15,Standard,15.49,2023-10-26,85.3,45.2,12,Drama,Smart TV,USA,25-34,False,2,15
1,usr_b8c7d6a5-f4e3-2d1c-0b9a-8c7d6a5f4e32,2018-11-20,Premium,19.99,2023-10-27,120.7,60.5,18,Action,Mobile,Canada,35-44,False,4,8
2,usr_1e2f3g4h-5i6j-7k8l-9m0n-1e2f3g4h5i6j,2022-03-01,Basic,9.99,2023-10-25,30.1,28.1,5,Comedy,Web Browser,UK,18-24,True,1,65
3,usr_a1b2c3d4-e5f6-7g8h-9i0j-a1b2c3d4e5f6,2021-08-03,Standard,15.49,2023-10-27,70.9,40.0,10,Sci-Fi,Smart TV,Australia,45-54,False,3,20
4,usr_f0e9d8c7-b6a5-4f3e-2d1c-0b9a8f7e6d5c,2019-01-10,Premium,19.99,2023-10-26,150.0,75.3,25,Thriller,Tablet,Germany,35-44,False,4,5




Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9f753d8844dd44ca9d.gradio.live
